# PyroFinder — YOLO11s Kaggle Training and Evaluation

This notebook trains and evaluates the planned primary PyroFinder detector, **YOLO11s**, on the **D-Fire** dataset.

**Project constraints**

- Detection classes are strictly `0 = smoke` and `1 = fire`.
- The provided D-Fire `train` and `test` split is preserved.
- YOLO11n is used only as the committed lightweight speed baseline/fallback for comparison.
- This notebook does not load or train YOLO11n.
- All YOLO11s metrics are generated from the completed run. No synthetic metrics or placeholder result values are used.
- Fire-location outputs are approximate normalized image-space measurements only, not precise geolocation.
- Nothing is uploaded to GitHub automatically.

**Kaggle prerequisites**

- Enable Internet so the notebook can clone the public repository and download `yolo11s.pt` when needed.
- Attach the D-Fire Kaggle dataset.
- Enable a GPU accelerator before running the training cell.

## 1. Run overview

This run fine-tunes YOLO11s on D-Fire using the same two-class task, image size, epoch count, batch size, and provided train/test split used by the measured YOLO11n baseline. The purpose is a fair YOLO11s-versus-YOLO11n comparison.

The notebook produces two separate result groups:

1. **Object-detection results:** mAP@0.5, mAP@0.5:0.95, Precision, Recall, F1, and per-class metrics when available.
2. **Operational-alert results:** Hazard Recall, False Alert Rate, Alert Precision, Alert F1, weighted error cost, Operational Alert Score, and approximate fire-location metrics.

The D-Fire `test` folder is used as the Ultralytics validation split, matching the existing baseline procedure.

## 2. GPU and environment check

This cell reports the environment but does not terminate the notebook when CUDA is unavailable. The training cell later enforces CUDA with a clear error so the notebook can still be inspected before the accelerator is enabled.

In [ ]:
from pathlib import Path
import importlib.metadata as importlib_metadata
import os
import platform
import sys

KAGGLE_WORKING_DIR = Path("/kaggle/working")
KAGGLE_INPUT_DIR = Path("/kaggle/input")

def package_version(distribution_name: str) -> str:
    try:
        return importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"

try:
    import torch
    TORCH_VERSION = torch.__version__
    CUDA_READY = bool(torch.cuda.is_available())
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_READY else "No CUDA GPU detected"
except Exception as exc:
    torch = None
    TORCH_VERSION = f"unavailable ({exc})"
    CUDA_READY = False
    GPU_NAME = "PyTorch could not inspect a GPU"

available_inputs = []
if KAGGLE_INPUT_DIR.exists():
    available_inputs = sorted(str(path) for path in KAGGLE_INPUT_DIR.iterdir())

print("Python version       :", sys.version.replace("\n", " "))
print("Platform             :", platform.platform())
print("PyTorch version      :", TORCH_VERSION)
print("Ultralytics version  :", package_version("ultralytics"))
print("CUDA available       :", CUDA_READY)
print("GPU name             :", GPU_NAME)
print("Current directory    :", Path.cwd())
print("Kaggle working dir   :", KAGGLE_WORKING_DIR)
print("Available inputs:")
if available_inputs:
    for item in available_inputs:
        print("  -", item)
else:
    print("  - No entries found under /kaggle/input")

if not CUDA_READY:
    print(
        "\nCUDA is not available. You may continue inspecting setup cells, "
        "but the YOLO11s training cell will stop until a Kaggle GPU accelerator is enabled."
    )

## 3. Install and import dependencies

Only dependencies required by this training/evaluation notebook are checked. The full repository `requirements.txt` also includes Streamlit and mapping packages that are not needed here, so they are not installed merely for this run.

Ultralytics is not imported in this setup cell; it is imported lazily only in training or inference/evaluation cells.

In [ ]:
import importlib.util
import re
import subprocess
import sys
from importlib import metadata as importlib_metadata

def numeric_version_tuple(version_text: str) -> tuple[int, ...]:
    numbers = re.findall(r"\d+", version_text)
    return tuple(int(value) for value in numbers[:3]) if numbers else (0,)

def ensure_dependency(
    module_name: str,
    install_spec: str,
    distribution_name: str,
    minimum_version: str | None = None,
) -> None:
    module_missing = importlib.util.find_spec(module_name) is None
    installed_version = None
    try:
        installed_version = importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        module_missing = True

    too_old = (
        minimum_version is not None
        and installed_version is not None
        and numeric_version_tuple(installed_version) < numeric_version_tuple(minimum_version)
    )

    if module_missing or too_old:
        reason = "missing" if module_missing else f"version {installed_version} is below {minimum_version}"
        print(f"Installing {install_spec} because {distribution_name} is {reason}.")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", install_spec],
            check=True,
        )
    else:
        print(f"{distribution_name} {installed_version} is already available; no installation needed.")

# Requirements relevant to this notebook, based on the repository requirements.txt.
ensure_dependency("ultralytics", "ultralytics>=8.3", "ultralytics", "8.3")
ensure_dependency("yaml", "PyYAML>=6", "PyYAML", "6")
ensure_dependency("pandas", "pandas>=2.2.3", "pandas")
ensure_dependency("matplotlib", "matplotlib", "matplotlib")
ensure_dependency("numpy", "numpy", "numpy")

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is missing. Use a standard Kaggle Python image with a GPU rather than "
        "installing a replacement torch build in this notebook."
    )

print("\nCore dependencies are ready. Ultralytics will be imported lazily later.")

## 4. Clone or refresh the repository

The repository is cloned into `/kaggle/working/pyrofinder`. On rerun, the cell fetches and hard-resets the existing clone to the current `origin/main`. Generated results stay outside the repository under `/kaggle/working/pyrofinder_yolo11s_results`.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Technion-LBS-Course/lisa-boris.git"
REPO_DIR = Path("/kaggle/working/pyrofinder")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Rename or remove it, then rerun this cell."
        )
    subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR, check=True)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

REPO_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()

required_project_files = [
    "PROJECT_CONTEXT.md",
    "CLAUDE.md",
    "README.md",
    "ASSISTANT_WORKING_RULES.md",
    "requirements.txt",
    "scripts/YOLO11n_baseline.py",
    "scripts/evaluate_yolo_alert_metrics.py",
    "src/evaluation.py",
    "results/baseline_yolo11n.json",
    "results/yolo11n_operational_metrics.json",
    "results/results_yolo11n.csv",
]
missing_project_files = [
    relative_path
    for relative_path in required_project_files
    if not (REPO_DIR / relative_path).exists()
]
if missing_project_files:
    raise FileNotFoundError(
        "The cloned repository is missing required source-of-truth files:\n- "
        + "\n- ".join(missing_project_files)
    )

print("Repository directory :", REPO_DIR)
print("Current commit hash  :", REPO_COMMIT)
print("Required project files validated.")

## 5. Validate D-Fire

Default input root:

`/kaggle/input/datasets/borisazarov/pyrofinder-dfire`

If that exact path is unavailable, the cell searches a bounded set of directories under `/kaggle/input` for a unique folder containing:

- `train/images`
- `train/labels`
- `test/images`
- `test/labels`

The run stops if the structure is invalid, ambiguous, incomplete, or does not match the expected D-Fire split sizes.

In [ ]:
from pathlib import Path

DEFAULT_DFIRE_ROOT = Path("/kaggle/input/datasets/borisazarov/pyrofinder-dfire")
REQUIRED_DFIRE_SUBDIRS = (
    Path("train/images"),
    Path("train/labels"),
    Path("test/images"),
    Path("test/labels"),
)
EXPECTED_TRAIN_IMAGES = 17_221
EXPECTED_TEST_IMAGES = 4_306
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_dfire_root(path: Path) -> bool:
    return path.is_dir() and all((path / subdir).is_dir() for subdir in REQUIRED_DFIRE_SUBDIRS)

def bounded_directory_search(root: Path, max_depth: int = 4) -> list[Path]:
    if not root.exists():
        return []
    seen = {root.resolve()}
    frontier = [root]
    found = []
    for _ in range(max_depth + 1):
        next_frontier = []
        for parent in frontier:
            if is_dfire_root(parent):
                found.append(parent.resolve())
                continue
            try:
                children = [child for child in parent.iterdir() if child.is_dir()]
            except PermissionError:
                children = []
            for child in children:
                resolved = child.resolve()
                if resolved not in seen:
                    seen.add(resolved)
                    next_frontier.append(child)
        frontier = next_frontier
    return sorted(set(found))

if is_dfire_root(DEFAULT_DFIRE_ROOT):
    DFIRE_ROOT = DEFAULT_DFIRE_ROOT
    print("Using default D-Fire root:", DFIRE_ROOT)
else:
    candidates = bounded_directory_search(Path("/kaggle/input"), max_depth=4)
    if not candidates:
        raise FileNotFoundError(
            "D-Fire was not found. Attach the Kaggle dataset and verify that one input folder "
            "contains train/images, train/labels, test/images, and test/labels. "
            f"Expected default: {DEFAULT_DFIRE_ROOT}"
        )
    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple D-Fire-like roots were found. Keep only the intended dataset input "
            "or set DFIRE_ROOT manually after reviewing these candidates:\n- "
            + "\n- ".join(str(path) for path in candidates)
        )
    DFIRE_ROOT = candidates[0]
    print("WARNING: The default D-Fire path was not found.")
    print("Discovered a valid D-Fire root instead:", DFIRE_ROOT)

def image_files(directory: Path) -> list[Path]:
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def label_files(directory: Path) -> list[Path]:
    return sorted(path for path in directory.iterdir() if path.is_file() and path.suffix.lower() == ".txt")

train_image_paths = image_files(DFIRE_ROOT / "train/images")
train_label_paths = label_files(DFIRE_ROOT / "train/labels")
test_image_paths = image_files(DFIRE_ROOT / "test/images")
test_label_paths = label_files(DFIRE_ROOT / "test/labels")

TRAIN_IMAGES_COUNT = len(train_image_paths)
TRAIN_LABELS_COUNT = len(train_label_paths)
TEST_IMAGES_COUNT = len(test_image_paths)
TEST_LABELS_COUNT = len(test_label_paths)

print(f"Train images: {TRAIN_IMAGES_COUNT:,}")
print(f"Train labels: {TRAIN_LABELS_COUNT:,}")
print(f"Test images : {TEST_IMAGES_COUNT:,}")
print(f"Test labels : {TEST_LABELS_COUNT:,}")

if TRAIN_IMAGES_COUNT != EXPECTED_TRAIN_IMAGES or TEST_IMAGES_COUNT != EXPECTED_TEST_IMAGES:
    raise RuntimeError(
        "D-Fire image counts do not match the verified project split. "
        f"Expected train={EXPECTED_TRAIN_IMAGES:,}, test={EXPECTED_TEST_IMAGES:,}; "
        f"found train={TRAIN_IMAGES_COUNT:,}, test={TEST_IMAGES_COUNT:,}. "
        "Do not continue until the intended dataset version is attached."
    )

train_image_stems = {path.stem for path in train_image_paths}
train_label_stems = {path.stem for path in train_label_paths}
test_image_stems = {path.stem for path in test_image_paths}
test_label_stems = {path.stem for path in test_label_paths}

missing_train_labels = sorted(train_image_stems - train_label_stems)
missing_test_labels = sorted(test_image_stems - test_label_stems)
orphan_train_labels = sorted(train_label_stems - train_image_stems)
orphan_test_labels = sorted(test_label_stems - test_image_stems)

if missing_train_labels or missing_test_labels or orphan_train_labels or orphan_test_labels:
    raise RuntimeError(
        "Image/label stem validation failed. "
        f"Missing train labels={len(missing_train_labels)}, "
        f"missing test labels={len(missing_test_labels)}, "
        f"orphan train labels={len(orphan_train_labels)}, "
        f"orphan test labels={len(orphan_test_labels)}."
    )

print("D-Fire structure, split sizes, and image/label alignment are valid.")

## 6. Build the YOLO11s data YAML

A Kaggle-local YAML file is created under the output root. The source dataset remains read-only and is not modified.

The class mapping is fixed:

- `0: smoke`
- `1: fire`

In [ ]:
from pathlib import Path
import yaml

OUTPUT_ROOT = Path("/kaggle/working/pyrofinder_yolo11s_results")
CONFIG_DIR = OUTPUT_ROOT / "config"
RESULTS_DIR = OUTPUT_ROOT / "results"
RUNS_DIR = OUTPUT_ROOT / "runs"

for directory in (OUTPUT_ROOT, CONFIG_DIR, RESULTS_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_YAML_PATH = CONFIG_DIR / "dfire_yolo11s.yaml"
data_yaml = {
    "path": DFIRE_ROOT.as_posix(),
    "train": "train/images",
    "val": "test/images",
    "nc": 2,
    "names": {0: "smoke", 1: "fire"},
}

with DATA_YAML_PATH.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(data_yaml, handle, sort_keys=False, allow_unicode=True)

print("Data YAML path:", DATA_YAML_PATH)
print(DATA_YAML_PATH.read_text(encoding="utf-8"))

## 7. Training configuration

The explicit settings match the measured YOLO11n baseline wherever applicable:

- Starting weights: `yolo11s.pt`
- Image size: `640`
- Epochs: `30`
- Batch size: `16`
- Device: CUDA device `0`
- Seed: `42`
- Dataset: D-Fire
- Train split: D-Fire `train`
- Validation/test split: D-Fire `test`
- Run name: `yolo11s_dfire_baseline`

Other training hyperparameters use the installed Ultralytics defaults and are preserved in the generated `args.yaml`. Automatic hyperparameter tuning is not used.

In [ ]:
from pprint import pprint

STARTING_WEIGHTS = "yolo11s.pt"
MODEL_NAME = "YOLO11s"
IMAGE_SIZE = 640
EPOCHS = 30
BATCH_SIZE = 16
DEVICE = 0
SEED = 42
RUN_NAME = "yolo11s_dfire_baseline"
RUN_DIR = RUNS_DIR / RUN_NAME

TRAIN_ARGS = {
    "data": str(DATA_YAML_PATH),
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH_SIZE,
    "device": DEVICE,
    "seed": SEED,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "plots": True,
    "save": True,
    "verbose": True,
}

CHECKPOINT_SELECTION_RULE = (
    "Use the Ultralytics-generated runs/yolo11s_dfire_baseline/weights/best.pt. "
    "Ultralytics selects best.pt using the installed version's validation fitness logic. "
    "last.pt is preserved but is not selected merely because it is the final epoch."
)

print("Training configuration:")
pprint({
    "model": MODEL_NAME,
    "starting_weights": STARTING_WEIGHTS,
    **TRAIN_ARGS,
    "dataset": "D-Fire",
    "train_split": "train",
    "validation_test_split": "test used as YOLO val",
    "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
})

## 8. Train YOLO11s

The training cell is safe to rerun after a completed run: if `best.pt` already exists, training is skipped unless the user deliberately prepares a new output directory.

If CUDA is unavailable, the cell stops before loading or training the model.

The selected checkpoint is always the generated `weights/best.pt`, copied to:

`/kaggle/working/pyrofinder_yolo11s_results/yolo11s_dfire_best.pt`

In [ ]:
import shutil
from pathlib import Path

if torch is None or not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Kaggle Notebook settings, enable a GPU accelerator, "
        "allow the session to restart, rerun the setup cells, and then rerun this cell."
    )

from ultralytics import YOLO  # Lazy import: heavy ML package is loaded only now.

RUN_BEST_PATH = RUN_DIR / "weights" / "best.pt"
RUN_LAST_PATH = RUN_DIR / "weights" / "last.pt"
SELECTED_BEST_PATH = OUTPUT_ROOT / "yolo11s_dfire_best.pt"
FORCE_RETRAIN = False

if RUN_BEST_PATH.exists() and not FORCE_RETRAIN:
    print("Completed checkpoint already exists; training is skipped:", RUN_BEST_PATH)
else:
    if FORCE_RETRAIN and RUN_DIR.exists():
        raise RuntimeError(
            f"FORCE_RETRAIN is True but {RUN_DIR} already exists. "
            "Rename or remove the existing run directory deliberately before retraining."
        )

    print("Starting YOLO11s training with batch size 16.")
    training_model = YOLO(STARTING_WEIGHTS)
    try:
        training_model.train(**TRAIN_ARGS)
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            print(
                "\nCUDA out-of-memory detected. The batch size was not changed automatically. "
                "Use the documented manual batch-size-8 fallback cell below."
            )
        raise

required_run_artifacts = [
    RUN_BEST_PATH,
    RUN_LAST_PATH,
    RUN_DIR / "results.csv",
    RUN_DIR / "args.yaml",
]
missing_run_artifacts = [path for path in required_run_artifacts if not path.exists()]
if missing_run_artifacts:
    raise FileNotFoundError(
        "Training did not produce all required artifacts:\n- "
        + "\n- ".join(str(path) for path in missing_run_artifacts)
    )

shutil.copy2(RUN_BEST_PATH, SELECTED_BEST_PATH)

print("Checkpoint selection rule:")
print(CHECKPOINT_SELECTION_RULE)
print("Selected source checkpoint:", RUN_BEST_PATH)
print("Copied selected checkpoint:", SELECTED_BEST_PATH)
print("Last checkpoint preserved  :", RUN_LAST_PATH)

### Manual CUDA out-of-memory fallback — batch size 8

This cell is disabled by default and must be enabled manually. It does not silently alter the primary configuration.

Use it only when the batch-size-16 training cell failed specifically because of CUDA out-of-memory:

1. Set `RUN_BATCH_8_FALLBACK = True`.
2. Rerun this cell.
3. Any incomplete run directory without `best.pt` is renamed and preserved before the batch-size-8 run starts.
4. The required final run name remains `yolo11s_dfire_baseline`.

In [ ]:
from datetime import datetime, timezone
import shutil

RUN_BATCH_8_FALLBACK = False

if not RUN_BATCH_8_FALLBACK:
    print("Batch-size-8 fallback is disabled. Set RUN_BATCH_8_FALLBACK = True only after a batch-16 OOM.")
else:
    if torch is None or not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable; the fallback also requires a Kaggle GPU.")
    if RUN_BEST_PATH.exists():
        raise RuntimeError(
            "A completed best.pt already exists. The fallback will not overwrite a completed run."
        )

    if RUN_DIR.exists():
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        preserved_incomplete_run = RUN_DIR.with_name(f"{RUN_NAME}_oom_incomplete_{timestamp}")
        RUN_DIR.rename(preserved_incomplete_run)
        print("Preserved incomplete batch-16 run at:", preserved_incomplete_run)

    fallback_args = dict(TRAIN_ARGS)
    fallback_args["batch"] = 8

    from ultralytics import YOLO
    fallback_model = YOLO(STARTING_WEIGHTS)
    fallback_model.train(**fallback_args)

    fallback_required = [
        RUN_BEST_PATH,
        RUN_LAST_PATH,
        RUN_DIR / "results.csv",
        RUN_DIR / "args.yaml",
    ]
    missing_fallback = [path for path in fallback_required if not path.exists()]
    if missing_fallback:
        raise FileNotFoundError(
            "Batch-size-8 fallback did not produce all required artifacts:\n- "
            + "\n- ".join(str(path) for path in missing_fallback)
        )

    shutil.copy2(RUN_BEST_PATH, SELECTED_BEST_PATH)
    print("Batch-size-8 fallback completed.")
    print("Selected checkpoint copied to:", SELECTED_BEST_PATH)

## 9. Object-detection evaluation

The selected YOLO11s `best.pt` checkpoint is evaluated on the D-Fire `test` folder through the YAML `val` split.

The result JSON is schema-compatible with the existing `baseline_yolo11n.json`. Missing metrics are written as `null` and reported as warnings rather than invented.

Outputs:

- `results/baseline_yolo11s.json`
- `results/results_yolo11s.csv`

In [ ]:
from datetime import date
import json
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

if not SELECTED_BEST_PATH.exists():
    raise FileNotFoundError(
        f"Selected YOLO11s checkpoint not found: {SELECTED_BEST_PATH}. Complete training first."
    )

from ultralytics import YOLO

EVAL_RUN_NAME = "yolo11s_dfire_test_evaluation"
evaluation_model = YOLO(str(SELECTED_BEST_PATH))
val_results = evaluation_model.val(
    data=str(DATA_YAML_PATH),
    imgsz=IMAGE_SIZE,
    split="val",
    device=DEVICE,
    project=str(RUNS_DIR),
    name=EVAL_RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
)

def tensor_or_array_to_list(value) -> list:
    if value is None:
        return []
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    try:
        array = np.asarray(value)
        return array.reshape(-1).tolist()
    except Exception:
        try:
            return list(value)
        except Exception:
            return []

def scalar_or_none(value, digits: int = 4):
    values = tensor_or_array_to_list(value)
    if not values:
        try:
            numeric = float(value)
        except Exception:
            return None
    else:
        try:
            numeric = float(values[0])
        except Exception:
            return None
    if not math.isfinite(numeric):
        return None
    return round(numeric, digits)

box_metrics = getattr(val_results, "box", None)
if box_metrics is None:
    print("WARNING: Ultralytics validation returned no box metrics object.")

map50 = scalar_or_none(getattr(box_metrics, "map50", None))
map50_95 = scalar_or_none(getattr(box_metrics, "map", None))
precision = scalar_or_none(getattr(box_metrics, "mp", None))
recall = scalar_or_none(getattr(box_metrics, "mr", None))
f1 = (
    round(2 * precision * recall / (precision + recall), 4)
    if precision is not None and recall is not None and (precision + recall) > 0
    else None
)

class_names = getattr(val_results, "names", {0: "smoke", 1: "fire"})
if isinstance(class_names, list):
    class_names = {index: name for index, name in enumerate(class_names)}

per_class = {}
if box_metrics is not None:
    class_indices = [int(value) for value in tensor_or_array_to_list(
        getattr(box_metrics, "ap_class_index", None)
    )]
    per_class_precision = tensor_or_array_to_list(getattr(box_metrics, "p", None))
    per_class_recall = tensor_or_array_to_list(getattr(box_metrics, "r", None))
    per_class_ap50 = tensor_or_array_to_list(getattr(box_metrics, "ap50", None))
    per_class_maps = tensor_or_array_to_list(getattr(box_metrics, "maps", None))

    for position, class_id in enumerate(class_indices):
        class_precision = (
            round(float(per_class_precision[position]), 4)
            if position < len(per_class_precision) else None
        )
        class_recall = (
            round(float(per_class_recall[position]), 4)
            if position < len(per_class_recall) else None
        )
        class_f1 = (
            round(2 * class_precision * class_recall / (class_precision + class_recall), 4)
            if class_precision is not None
            and class_recall is not None
            and (class_precision + class_recall) > 0
            else None
        )
        class_map50 = (
            round(float(per_class_ap50[position]), 4)
            if position < len(per_class_ap50) else None
        )
        class_map50_95 = (
            round(float(per_class_maps[class_id]), 4)
            if class_id < len(per_class_maps) else None
        )
        class_name = str(class_names.get(class_id, f"class_{class_id}"))
        per_class[class_name] = {
            "map50": class_map50,
            "map50_95": class_map50_95,
            "precision": class_precision,
            "recall": class_recall,
            "f1": class_f1,
        }

detection_metrics = {
    "map50": map50,
    "map50_95": map50_95,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "per_class": per_class or None,
    "metrics_source": (
        f"Ultralytics validation of {SELECTED_BEST_PATH} using {DATA_YAML_PATH}; "
        "split=val, where YAML val points to the D-Fire test/images folder."
    ),
    "selected_epoch_or_row": None,
}

for metric_name in ("map50", "map50_95", "precision", "recall", "f1"):
    if detection_metrics[metric_name] is None:
        print(f"WARNING: {metric_name} could not be extracted and will be saved as null.")

training_results_source = RUN_DIR / "results.csv"
training_results_copy = RESULTS_DIR / "results_yolo11s.csv"
if not training_results_source.exists():
    raise FileNotFoundError(f"Training CSV not found: {training_results_source}")
shutil.copy2(training_results_source, training_results_copy)

args_yaml_path = RUN_DIR / "args.yaml"
training_args_record = yaml.safe_load(args_yaml_path.read_text(encoding="utf-8")) or {}

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return value

validation_speed = json_safe(getattr(val_results, "speed", None))
actual_batch = training_args_record.get("batch")
actual_epochs = training_args_record.get("epochs", EPOCHS)

baseline_yolo11s = {
    "model_name": "YOLO11s (object detection candidate)",
    "model_family": "object_detection",
    "run_date": date.today().isoformat(),
    "status": "completed_measured",
    "synthetic_metrics": False,
    "task": "two-class object detection: fire / smoke",
    "repository_commit": REPO_COMMIT,
    "dataset": {
        "name": "D-Fire",
        "source_root": str(DFIRE_ROOT),
        "train_size": TRAIN_IMAGES_COUNT,
        "test_size": TEST_IMAGES_COUNT,
        "class_mapping": {"0": "smoke", "1": "fire"},
        "label_format": "YOLO normalized bounding boxes",
        "evaluation_split": "D-Fire test folder used as YOLO val split",
    },
    "model_params": {
        "model": "YOLO11s",
        "starting_weights": STARTING_WEIGHTS,
        "trained_weights": str(SELECTED_BEST_PATH),
        "checkpoint_source": str(RUN_BEST_PATH),
        "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
        "imgsz": IMAGE_SIZE,
        "epochs_requested": EPOCHS,
        "epochs_recorded_in_args_yaml": actual_epochs,
        "selected_epoch_or_row": None,
        "batch_requested": BATCH_SIZE,
        "batch_recorded_in_args_yaml": actual_batch,
        "device": DEVICE,
        "seed": SEED,
        "run_dir": str(RUN_DIR),
        "training_args": json_safe(training_args_record),
    },
    "metrics": detection_metrics,
    "validation_speed_ms": validation_speed,
    "notes": (
        "Measured YOLO11s object-detection results generated from the selected best.pt "
        "checkpoint. No synthetic values are used. Detection metrics and operational "
        "alert metrics are evaluated separately."
    ),
}

BASELINE_YOLO11S_JSON = RESULTS_DIR / "baseline_yolo11s.json"
BASELINE_YOLO11S_JSON.write_text(
    json.dumps(baseline_yolo11s, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved detection JSON :", BASELINE_YOLO11S_JSON)
print("Copied training CSV  :", training_results_copy)
print("Detection metrics:")
print(json.dumps(detection_metrics, indent=2))

## 10. Operational alert evaluation

The existing repository script `scripts/evaluate_yolo_alert_metrics.py` runs inference only. It does not retrain the model.

Configuration:

- Model name: `YOLO11s`
- Confidence threshold: `0.25`
- Image size: `640`
- False-negative weight: `10`
- False-positive weight: `1`

Outputs:

- `results/yolo11s_operational_metrics.json`
- `results/yolo11s_test_predictions.csv`

In [ ]:
import subprocess
import sys

OPERATIONAL_JSON = RESULTS_DIR / "yolo11s_operational_metrics.json"
OPERATIONAL_CSV = RESULTS_DIR / "yolo11s_test_predictions.csv"
OPERATIONAL_SCRIPT = REPO_DIR / "scripts/evaluate_yolo_alert_metrics.py"

if not OPERATIONAL_SCRIPT.exists():
    raise FileNotFoundError(f"Operational evaluation script not found: {OPERATIONAL_SCRIPT}")
if not SELECTED_BEST_PATH.exists():
    raise FileNotFoundError(f"Selected checkpoint not found: {SELECTED_BEST_PATH}")

operational_command = [
    sys.executable,
    str(OPERATIONAL_SCRIPT),
    "--raw-root", str(DFIRE_ROOT),
    "--weights", str(SELECTED_BEST_PATH),
    "--model-name", "YOLO11s",
    "--conf", "0.25",
    "--imgsz", str(IMAGE_SIZE),
    "--fn-weight", "10",
    "--fp-weight", "1",
    "--output-json", str(OPERATIONAL_JSON),
    "--output-csv", str(OPERATIONAL_CSV),
    "--device", str(DEVICE),
]

print("Running inference-only operational evaluation:")
print(" ".join(operational_command))
subprocess.run(operational_command, cwd=REPO_DIR, check=True)

for output_file in (OPERATIONAL_JSON, OPERATIONAL_CSV):
    if not output_file.exists() or output_file.stat().st_size == 0:
        raise FileNotFoundError(f"Operational evaluation output is missing or empty: {output_file}")

print("Operational evaluation completed without retraining.")
print("JSON:", OPERATIONAL_JSON)
print("CSV :", OPERATIONAL_CSV)

## 11. Compare YOLO11s to the measured YOLO11n baseline

The committed measured YOLO11n files are loaded at runtime:

- `results/baseline_yolo11n.json`
- `results/yolo11n_operational_metrics.json`

Detection and operational results are shown in separate tables because they answer different questions.

**Measured quality promotion gate**

YOLO11s passes the quality gate only when:

1. its detection and operational outputs exist and are explicitly marked as measured/non-synthetic;
2. all required comparison metrics are available;
3. YOLO11s improves both mAP@0.5 and detection Recall over YOLO11n;
4. YOLO11s does not reduce Hazard Recall;
5. YOLO11s does not increase False Alert Rate;
6. YOLO11s does not reduce Operational Alert Score.

Even after a quality-gate pass, YOLO11n remains the lightweight speed baseline/fallback. A final deployment choice should also use an equal-hardware inference-speed comparison.

In [ ]:
import json
import pandas as pd
from IPython.display import display

YOLO11N_DETECTION_JSON = REPO_DIR / "results/baseline_yolo11n.json"
YOLO11N_OPERATIONAL_JSON = REPO_DIR / "results/yolo11n_operational_metrics.json"

for source_file in (
    YOLO11N_DETECTION_JSON,
    YOLO11N_OPERATIONAL_JSON,
    BASELINE_YOLO11S_JSON,
    OPERATIONAL_JSON,
):
    if not source_file.exists() or source_file.stat().st_size == 0:
        raise FileNotFoundError(f"Required measured result file is missing or empty: {source_file}")

yolo11n_detection = json.loads(YOLO11N_DETECTION_JSON.read_text(encoding="utf-8"))
yolo11n_operational = json.loads(YOLO11N_OPERATIONAL_JSON.read_text(encoding="utf-8"))
yolo11s_detection = json.loads(BASELINE_YOLO11S_JSON.read_text(encoding="utf-8"))
yolo11s_operational = json.loads(OPERATIONAL_JSON.read_text(encoding="utf-8"))

n_det = yolo11n_detection.get("metrics", {})
s_det = yolo11s_detection.get("metrics", {})
n_op = yolo11n_operational.get("operational_metrics", {})
s_op = yolo11s_operational.get("operational_metrics", {})
n_loc = yolo11n_operational.get("location_metrics", {})
s_loc = yolo11s_operational.get("location_metrics", {})

detection_comparison = pd.DataFrame([
    {
        "Model": "YOLO11n",
        "Role": "Lightweight speed baseline/fallback",
        "mAP@0.5": n_det.get("map50"),
        "mAP@0.5:0.95": n_det.get("map50_95"),
        "Precision": n_det.get("precision"),
        "Recall": n_det.get("recall"),
        "F1": n_det.get("f1"),
    },
    {
        "Model": "YOLO11s",
        "Role": "Planned primary detector candidate",
        "mAP@0.5": s_det.get("map50"),
        "mAP@0.5:0.95": s_det.get("map50_95"),
        "Precision": s_det.get("precision"),
        "Recall": s_det.get("recall"),
        "F1": s_det.get("f1"),
    },
])

operational_comparison = pd.DataFrame([
    {
        "Model": "YOLO11n",
        "Role": "Lightweight speed baseline/fallback",
        "Hazard Recall": n_op.get("hazard_recall"),
        "False Alert Rate": n_op.get("false_alert_rate"),
        "Alert Precision": n_op.get("alert_precision"),
        "Alert F1": n_op.get("alert_f1"),
        "Operational Alert Score": n_op.get("operational_alert_score"),
        "Location Coverage": n_loc.get("location_coverage_rate"),
        "3x3 Grid Hit Rate": n_loc.get("fire_location_grid_hit_rate"),
    },
    {
        "Model": "YOLO11s",
        "Role": "Planned primary detector candidate",
        "Hazard Recall": s_op.get("hazard_recall"),
        "False Alert Rate": s_op.get("false_alert_rate"),
        "Alert Precision": s_op.get("alert_precision"),
        "Alert F1": s_op.get("alert_f1"),
        "Operational Alert Score": s_op.get("operational_alert_score"),
        "Location Coverage": s_loc.get("location_coverage_rate"),
        "3x3 Grid Hit Rate": s_loc.get("fire_location_grid_hit_rate"),
    },
])

print("Object-detection comparison")
display(detection_comparison)

print("\nOperational-alert comparison")
display(operational_comparison)

required_detection_values = [
    s_det.get("map50"),
    s_det.get("map50_95"),
    s_det.get("precision"),
    s_det.get("recall"),
    s_det.get("f1"),
]
required_operational_values = [
    s_op.get("hazard_recall"),
    s_op.get("false_alert_rate"),
    s_op.get("alert_precision"),
    s_op.get("alert_f1"),
    s_op.get("operational_alert_score"),
]

measured_outputs_valid = (
    yolo11s_detection.get("status") == "completed_measured"
    and yolo11s_detection.get("synthetic_metrics") is False
    and yolo11s_operational.get("evaluation_type") == "operational_alert_metrics"
    and yolo11s_operational.get("dataset", {}).get("num_images_evaluated") == TEST_IMAGES_COUNT
)
all_metrics_available = all(
    value is not None for value in required_detection_values + required_operational_values
)

quality_gate_checks = {
    "Measured YOLO11s outputs are valid": measured_outputs_valid,
    "All required YOLO11s metrics are available": all_metrics_available,
    "YOLO11s mAP@0.5 improves": (
        s_det.get("map50") is not None
        and n_det.get("map50") is not None
        and s_det["map50"] > n_det["map50"]
    ),
    "YOLO11s detection Recall improves": (
        s_det.get("recall") is not None
        and n_det.get("recall") is not None
        and s_det["recall"] > n_det["recall"]
    ),
    "YOLO11s Hazard Recall does not regress": (
        s_op.get("hazard_recall") is not None
        and n_op.get("hazard_recall") is not None
        and s_op["hazard_recall"] >= n_op["hazard_recall"]
    ),
    "YOLO11s False Alert Rate does not increase": (
        s_op.get("false_alert_rate") is not None
        and n_op.get("false_alert_rate") is not None
        and s_op["false_alert_rate"] <= n_op["false_alert_rate"]
    ),
    "YOLO11s Operational Alert Score does not regress": (
        s_op.get("operational_alert_score") is not None
        and n_op.get("operational_alert_score") is not None
        and s_op["operational_alert_score"] >= n_op["operational_alert_score"]
    ),
}

quality_gate_table = pd.DataFrame([
    {"Gate": gate, "Passed": passed}
    for gate, passed in quality_gate_checks.items()
])
print("\nMeasured quality-promotion gate")
display(quality_gate_table)

if all(quality_gate_checks.values()):
    print(
        "Result: YOLO11s passes the measured quality gate. "
        "YOLO11n remains the lightweight speed baseline/fallback. "
        "Complete an equal-hardware inference-speed check before the final deployment selection."
    )
else:
    print(
        "Result: YOLO11s does not pass every measured quality gate. "
        "Do not declare it the winner; retain YOLO11n as the current fallback and review the failed gates."
    )

## 12. Training-curve review

Separate plots are generated for:

1. training and validation losses;
2. validation mAP;
3. Precision;
4. Recall.

The diagnostic is a transparent heuristic based only on the observed CSV curves. It is not a statistical proof of overfitting.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

training_curve_df = pd.read_csv(RESULTS_DIR / "results_yolo11s.csv")
training_curve_df.columns = [column.strip() for column in training_curve_df.columns]
if "epoch" not in training_curve_df.columns:
    raise KeyError("The training CSV has no 'epoch' column.")

CURVE_DIR = OUTPUT_ROOT / "curve_review"
CURVE_DIR.mkdir(parents=True, exist_ok=True)

loss_columns = [
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
]
available_loss_columns = [column for column in loss_columns if column in training_curve_df.columns]
if not available_loss_columns:
    raise KeyError("No expected training or validation loss columns were found.")

fig, ax = plt.subplots(figsize=(11, 6))
for column in available_loss_columns:
    ax.plot(training_curve_df["epoch"], training_curve_df[column], label=column)
ax.set_title("YOLO11s training and validation losses")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
loss_plot_path = CURVE_DIR / "training_validation_losses.png"
fig.savefig(loss_plot_path, dpi=160, bbox_inches="tight")
plt.show()

map_columns = [
    column for column in ("metrics/mAP50(B)", "metrics/mAP50-95(B)")
    if column in training_curve_df.columns
]
if not map_columns:
    raise KeyError("No validation mAP columns were found.")

fig, ax = plt.subplots(figsize=(11, 5))
for column in map_columns:
    ax.plot(training_curve_df["epoch"], training_curve_df[column], label=column)
ax.set_title("YOLO11s validation mAP")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
map_plot_path = CURVE_DIR / "validation_map.png"
fig.savefig(map_plot_path, dpi=160, bbox_inches="tight")
plt.show()

precision_column = "metrics/precision(B)"
if precision_column not in training_curve_df.columns:
    raise KeyError(f"Missing precision column: {precision_column}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(training_curve_df["epoch"], training_curve_df[precision_column])
ax.set_title("YOLO11s validation Precision")
ax.set_xlabel("Epoch")
ax.set_ylabel("Precision")
ax.grid(True, alpha=0.3)
fig.tight_layout()
precision_plot_path = CURVE_DIR / "validation_precision.png"
fig.savefig(precision_plot_path, dpi=160, bbox_inches="tight")
plt.show()

recall_column = "metrics/recall(B)"
if recall_column not in training_curve_df.columns:
    raise KeyError(f"Missing recall column: {recall_column}")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(training_curve_df["epoch"], training_curve_df[recall_column])
ax.set_title("YOLO11s validation Recall")
ax.set_xlabel("Epoch")
ax.set_ylabel("Recall")
ax.grid(True, alpha=0.3)
fig.tight_layout()
recall_plot_path = CURVE_DIR / "validation_recall.png"
fig.savefig(recall_plot_path, dpi=160, bbox_inches="tight")
plt.show()

train_loss_available = [
    column for column in ("train/box_loss", "train/cls_loss", "train/dfl_loss")
    if column in training_curve_df.columns
]
val_loss_available = [
    column for column in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
    if column in training_curve_df.columns
]
map95_column = "metrics/mAP50-95(B)"

train_loss_total = training_curve_df[train_loss_available].sum(axis=1)
val_loss_total = training_curve_df[val_loss_available].sum(axis=1)
map95_series = training_curve_df[map95_column]

tail_length = max(5, len(training_curve_df) // 3)
tail_x = np.arange(tail_length)
train_loss_slope = float(np.polyfit(tail_x, train_loss_total.tail(tail_length), 1)[0])
val_loss_slope = float(np.polyfit(tail_x, val_loss_total.tail(tail_length), 1)[0])
peak_map95 = float(map95_series.max())
peak_epoch = int(training_curve_df.loc[map95_series.idxmax(), "epoch"])
final_map95 = float(map95_series.iloc[-1])
map95_drop = peak_map95 - final_map95

if train_loss_slope < 0 and val_loss_slope > 0 and map95_drop > 0.02:
    diagnostic = (
        "Possible overfitting signal: training loss continued downward while validation loss "
        "trended upward, and final mAP@0.5:0.95 fell materially below its observed peak."
    )
elif map95_drop <= 0.01 and val_loss_slope <= 0:
    diagnostic = (
        "No strong overfitting signal in this heuristic: final mAP@0.5:0.95 remained close "
        "to its peak and validation loss did not trend upward over the final portion."
    )
else:
    diagnostic = (
        "Mixed or inconclusive curve behavior: the observed loss and mAP trends do not meet "
        "the notebook's threshold for either a clear overfitting signal or a clear stable pattern."
    )

print("Curve-based overfitting diagnostic")
print("------------------------------------")
print(f"Peak mAP@0.5:0.95 : {peak_map95:.6f} at epoch {peak_epoch}")
print(f"Final mAP@0.5:0.95: {final_map95:.6f}")
print(f"Peak-to-final drop : {map95_drop:.6f}")
print(f"Final-tail train-loss slope: {train_loss_slope:.6f}")
print(f"Final-tail val-loss slope  : {val_loss_slope:.6f}")
print(diagnostic)
print("\nThis is a curve heuristic only; inspect the displayed plots before drawing a final conclusion.")

## 13. Output validation

This stage verifies that every required output exists and is non-empty, then reports paths, sizes, JSON keys, CSV shapes, and final measured metrics.

In [ ]:
import json
from pathlib import Path
import pandas as pd

required_outputs = {
    "Selected best checkpoint": SELECTED_BEST_PATH,
    "Training run best.pt": RUN_BEST_PATH,
    "Training run last.pt": RUN_LAST_PATH,
    "Training args.yaml": RUN_DIR / "args.yaml",
    "Training results.csv": RUN_DIR / "results.csv",
    "YOLO11s detection JSON": BASELINE_YOLO11S_JSON,
    "YOLO11s copied training CSV": RESULTS_DIR / "results_yolo11s.csv",
    "YOLO11s operational JSON": OPERATIONAL_JSON,
    "YOLO11s per-image predictions CSV": OPERATIONAL_CSV,
    "Loss plot": CURVE_DIR / "training_validation_losses.png",
    "mAP plot": CURVE_DIR / "validation_map.png",
    "Precision plot": CURVE_DIR / "validation_precision.png",
    "Recall plot": CURVE_DIR / "validation_recall.png",
}

validation_failures = []
for label, path in required_outputs.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    print(f"{label:38s} | {path} | {size:,} bytes")
    if not exists or size == 0:
        validation_failures.append(f"{label}: {path}")

if validation_failures:
    raise FileNotFoundError(
        "Required outputs are missing or empty:\n- " + "\n- ".join(validation_failures)
    )

detection_doc = json.loads(BASELINE_YOLO11S_JSON.read_text(encoding="utf-8"))
operational_doc = json.loads(OPERATIONAL_JSON.read_text(encoding="utf-8"))
training_csv = pd.read_csv(RESULTS_DIR / "results_yolo11s.csv")
prediction_csv = pd.read_csv(OPERATIONAL_CSV)

print("\nDetection JSON top-level keys:", sorted(detection_doc.keys()))
print("Operational JSON top-level keys:", sorted(operational_doc.keys()))
print("Training CSV shape:", training_csv.shape)
print("Prediction CSV shape:", prediction_csv.shape)

print("\nFinal YOLO11s detection metrics:")
print(json.dumps(detection_doc.get("metrics", {}), indent=2))

print("\nFinal YOLO11s operational metrics:")
print(json.dumps({
    "operational_metrics": operational_doc.get("operational_metrics"),
    "location_metrics": operational_doc.get("location_metrics"),
}, indent=2))

print("\nAll required output validations passed.")

## 14. Create the downloadable ZIP

The ZIP contains the selected checkpoint, result JSON/CSV files, training run metadata, standard Ultralytics plots/logs, the notebook-generated curve-review plots, and a plain-text run summary.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import zipfile

RUN_SUMMARY_PATH = OUTPUT_ROOT / "run_summary.txt"
ZIP_PATH = Path("/kaggle/working/pyrofinder_yolo11s_results.zip")

detection_doc = json.loads(BASELINE_YOLO11S_JSON.read_text(encoding="utf-8"))
operational_doc = json.loads(OPERATIONAL_JSON.read_text(encoding="utf-8"))

summary_lines = [
    "PyroFinder YOLO11s D-Fire training and evaluation",
    "================================================",
    f"Created UTC: {datetime.now(timezone.utc).isoformat()}",
    f"Repository commit: {REPO_COMMIT}",
    f"D-Fire root: {DFIRE_ROOT}",
    f"Train images: {TRAIN_IMAGES_COUNT}",
    f"Test images: {TEST_IMAGES_COUNT}",
    "Class mapping: 0=smoke, 1=fire",
    f"Starting weights: {STARTING_WEIGHTS}",
    f"Selected checkpoint: {SELECTED_BEST_PATH}",
    f"Checkpoint source: {RUN_BEST_PATH}",
    f"Checkpoint rule: {CHECKPOINT_SELECTION_RULE}",
    "",
    "Detection metrics:",
    json.dumps(detection_doc.get("metrics", {}), indent=2),
    "",
    "Operational metrics:",
    json.dumps(operational_doc.get("operational_metrics", {}), indent=2),
    "",
    "Approximate image-space location metrics:",
    json.dumps(operational_doc.get("location_metrics", {}), indent=2),
    "",
    "No synthetic metrics were used.",
    "No files were uploaded to GitHub automatically.",
]
RUN_SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUTPUT_ROOT))

if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size == 0:
    raise RuntimeError(f"ZIP creation failed: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, mode="r") as archive:
    archived_names = set(archive.namelist())

minimum_archive_entries = {
    "yolo11s_dfire_best.pt",
    "results/baseline_yolo11s.json",
    "results/results_yolo11s.csv",
    "results/yolo11s_operational_metrics.json",
    "results/yolo11s_test_predictions.csv",
    "runs/yolo11s_dfire_baseline/args.yaml",
    "runs/yolo11s_dfire_baseline/results.csv",
    "curve_review/training_validation_losses.png",
    "curve_review/validation_map.png",
    "curve_review/validation_precision.png",
    "curve_review/validation_recall.png",
    "run_summary.txt",
}
missing_archive_entries = sorted(minimum_archive_entries - archived_names)
if missing_archive_entries:
    raise RuntimeError(
        "ZIP is missing required entries:\n- " + "\n- ".join(missing_archive_entries)
    )

print("ZIP validation passed.")
print("Download this file from Kaggle Outputs:")
print(ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size:,} bytes")

## 15. Save the Kaggle version and preserve outputs

Kaggle `/kaggle/working` storage is temporary. After every cell completes successfully:

1. Click **Save Version**.
2. Choose **Save & Run All**.
3. Keep notebook output files enabled.
4. Wait for the saved version to finish.
5. Open the version's **Output** section.
6. Download `/kaggle/working/pyrofinder_yolo11s_results.zip`.

Do not close the session before verifying that the ZIP exists and the output-validation cell passed.